# Slingshot QC: cross-validate DPT pseudotime per lineage

Independent corroboration that the DPT trajectories computed in `260502-pseudotime_inhibitory.ipynb` and `260502-pseudotime_excitatory.ipynb` are well-defined.

Method: for each lineage, run a pure-Python slingshot implementation on the global UMAP coordinates of the UMI dataset (subsetted to the lineage), then compute the Spearman correlation against the DPT pseudotime stored in the lineage checkpoint.

Pass criterion: Spearman rho (DPT vs slingshot) >= 0.85 per lineage. Lower values indicate that the trajectory may have substructure not captured by either method, and warrant deeper investigation before downstream regression.

The `slingshot_curve` function is reused verbatim from `3UTR/organoids_knoblich/UTR/260502-plot_UMAP.ipynb` (cell labelled `# Slingshot-inspired single-lineage principal curve`).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UMI")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from scipy.interpolate import CubicSpline
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors
import warnings

warnings.filterwarnings("ignore")

LAYOUT_DPI = 300
FONT_LABEL = 7
FONT_TICK = 6
FONT_LEGEND = 5.5

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": LAYOUT_DPI,
        "font.size": FONT_LABEL,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": FONT_LABEL,
        "axes.labelsize": FONT_LABEL,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
sc.settings.verbosity = 0
print("Setup complete.")

In [ ]:
LINEAGES = {
    "Inhibitory neurons": [
        "vRG",
        "ccvRG",
        "INP",
        "LGE_IN",
        "CGE_LGE_IN",
        "CGE_IN",
    ],
    "Excitatory neurons": ["oRG", "RG", "ccRG", "IP", "L6_CThPN", "L56", "L4", "L23"],
}
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
PASS_RHO = 0.85

BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich"
COUNTS_PATH = f"{BASEPATH}/cellranger"
COUNTS_FILE = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized.h5ad"
LIN_FILES = {
    lin: f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized_{tag}_dpt.h5ad"
    for lin, tag in LINEAGE_TAG.items()
}

FIG_DIR = Path(f"{COUNTS_PATH}/figures/pseudotime_slingshot_QC")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures: {FIG_DIR}")

## Slingshot helper

Pure-Python slingshot: principal curve through cluster centroids via MST + cubic spline. Source: `260502-plot_UMAP.ipynb`.


In [ ]:
def slingshot_curve(umap_coords_lin, obs_lin, celltype_col="celltype"):
    """Slingshot-inspired single-lineage principal curve (Street et al. 2018).

    Spline is fit through CLUSTER CENTROIDS rather than individual cells. The MST
    over centroids provides an unbiased ordering of waypoints without requiring
    prior biological knowledge of cell-type order.

    Returns (t, curve, waypts):
      t      : (n,) arc-length pseudotime per cell
      curve  : (500, 2) smooth curve for plotting
      waypts : (k, 2) ordered centroid waypoints
    """
    cell_types = obs_lin[celltype_col].values

    unique_cts = (
        obs_lin[celltype_col].cat.remove_unused_categories().cat.categories.tolist()
    )
    centroids = np.array(
        [umap_coords_lin[cell_types == ct].mean(axis=0) for ct in unique_cts]
    )
    k = len(unique_cts)

    diff = centroids[:, None, :] - centroids[None, :, :]
    dist_mat = np.linalg.norm(diff, axis=-1)
    mst = minimum_spanning_tree(csr_matrix(dist_mat))
    mst_dense = (mst + mst.T).toarray()

    cloud_center = umap_coords_lin.mean(axis=0)
    degrees = (mst_dense > 0).sum(axis=1)
    leaf_indices = np.where(degrees == 1)[0]
    if len(leaf_indices) == 0:
        leaf_indices = np.arange(k)
    root_idx = leaf_indices[
        np.argmin(np.linalg.norm(centroids[leaf_indices] - cloud_center, axis=1))
    ]

    visited = [False] * k
    path = []

    def dfs(node):
        visited[node] = True
        path.append(node)
        for nbr in range(k):
            if mst_dense[node, nbr] > 0 and not visited[nbr]:
                dfs(nbr)

    dfs(root_idx)
    waypts = centroids[path]

    diffs = np.diff(waypts, axis=0)
    seg_lengths = np.linalg.norm(diffs, axis=1)
    t_waypts = np.concatenate([[0.0], np.cumsum(seg_lengths)])
    cs_x = CubicSpline(t_waypts, waypts[:, 0])
    cs_y = CubicSpline(t_waypts, waypts[:, 1])

    t_fine = np.linspace(0.0, t_waypts[-1], 500)
    curve = np.column_stack([cs_x(t_fine), cs_y(t_fine)])

    arc_diffs = np.diff(curve, axis=0)
    arc = np.concatenate([[0.0], np.cumsum(np.linalg.norm(arc_diffs, axis=1))])

    nbrs = NearestNeighbors(n_neighbors=1).fit(curve)
    _, idx = nbrs.kneighbors(umap_coords_lin)
    t = arc[idx.ravel()]

    return t, curve, waypts

## Load data


In [ ]:
adata_full = sc.read_h5ad(COUNTS_FILE)
print(f"Full UMI: {adata_full.shape}, X_umap shape: {adata_full.obsm['X_umap'].shape}")

adata_lin = {}
for lin, path in LIN_FILES.items():
    adata_lin[lin] = sc.read_h5ad(path)
    print(
        f"{lin}: {adata_lin[lin].shape}, dpt range "
        f"[{adata_lin[lin].obs['dpt_pseudotime'].min():.3f}, "
        f"{adata_lin[lin].obs['dpt_pseudotime'].max():.3f}]"
    )

## Run slingshot per lineage and compare against DPT


In [ ]:
results = {}

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for row, (lineage_name, canonical) in enumerate(LINEAGES.items()):
    ad_lin_dpt = adata_lin[lineage_name]

    lin_mask = (adata_full.obs["lineage"] == lineage_name).values
    umap_lin = adata_full.obsm["X_umap"][lin_mask]
    obs_lin = adata_full.obs[lin_mask].copy()
    obs_lin["celltype"] = obs_lin["celltype"].cat.remove_unused_categories()
    obs_names_full = adata_full.obs_names[lin_mask]

    sling_t, curve, waypts = slingshot_curve(umap_lin.astype(float), obs_lin)

    valid_cts = [ct for ct in canonical if (obs_lin["celltype"] == ct).any()]
    ct_means = {
        ct: sling_t[(obs_lin["celltype"] == ct).values].mean() for ct in valid_cts
    }
    rho_canon, _ = spearmanr(range(len(valid_cts)), [ct_means[ct] for ct in valid_cts])
    if rho_canon < 0:
        sling_t = sling_t.max() - sling_t

    dpt_lookup = pd.Series(
        ad_lin_dpt.obs["dpt_pseudotime"].values, index=ad_lin_dpt.obs_names
    )
    dpt_aligned = dpt_lookup.reindex(obs_names_full).values
    valid = ~np.isnan(dpt_aligned)
    rho_compare, p_compare = spearmanr(dpt_aligned[valid], sling_t[valid])

    results[lineage_name] = {
        "n_cells": int(valid.sum()),
        "rho_canonical_slingshot": float(rho_canon),
        "rho_dpt_vs_slingshot": float(rho_compare),
        "p_dpt_vs_slingshot": float(p_compare),
        "passes": float(rho_compare) >= PASS_RHO,
    }

    ax_umap = axes[row, 0]
    for ct in valid_cts:
        m = (obs_lin["celltype"] == ct).values
        ax_umap.scatter(
            umap_lin[m, 0], umap_lin[m, 1], s=2, alpha=0.5, label=ct, rasterized=True
        )
    ax_umap.plot(
        curve[:, 0], curve[:, 1], color="black", lw=1.5, zorder=10, label="Slingshot"
    )
    ax_umap.scatter(
        waypts[:, 0],
        waypts[:, 1],
        c="black",
        s=30,
        zorder=11,
        marker="D",
        linewidths=0.5,
        edgecolors="white",
    )
    ax_umap.set_xlabel("UMAP 1")
    ax_umap.set_ylabel("UMAP 2")
    ax_umap.set_title(f"{lineage_name}: slingshot on global UMAP")
    ax_umap.legend(fontsize=FONT_LEGEND, markerscale=2, frameon=False, loc="best")

    ax_sc = axes[row, 1]
    cmap = plt.get_cmap("tab10")
    cat_codes = obs_lin["celltype"].cat.codes.values
    sc_obj = ax_sc.scatter(
        dpt_aligned[valid],
        sling_t[valid],
        c=cat_codes[valid],
        cmap=cmap,
        s=2,
        alpha=0.5,
        rasterized=True,
    )
    ax_sc.set_xlabel("DPT pseudotime")
    ax_sc.set_ylabel("Slingshot pseudotime")
    ax_sc.set_title(
        f"{lineage_name}: rho={rho_compare:.3f} (p={p_compare:.2e}) "
        f"{'PASS' if rho_compare >= PASS_RHO else 'CHECK'}"
    )

plt.tight_layout()
plt.savefig(FIG_DIR / "slingshot_vs_dpt.pdf", bbox_inches="tight")
plt.show()

print()
print("Summary:")
for lin, r in results.items():
    print(
        f"  {lin}: n={r['n_cells']:,} cells, "
        f"rho(DPT,slingshot)={r['rho_dpt_vs_slingshot']:.3f}, "
        f"{'PASS' if r['passes'] else 'CHECK'}"
    )

## Outcome

If both lineages have rho >= 0.85, DPT is corroborated and the lineage checkpoints are ready for propagation by `260502-propagate_pseudotime_to_UTR.ipynb`. Lower values do not invalidate DPT (slingshot operates on 2D UMAP, DPT on diffmap, they need not match perfectly), but warrant a second look at the QC plots in the per-lineage DPT notebooks before proceeding.
